# 02 - Preprocessing and PCA

This notebook uses the reusable preprocessing and PCA modules to transform the Paddy Dataset into a model-ready feature matrix, reduce dimensionality with PCA, and save outputs for clustering and visualization.

## Setup

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.preprocessing import load_paddy_dataset, preprocess_paddy_dataset, save_preprocessing_outputs
from src.pca_analysis import fit_pca, save_pca_outputs

DATA_PATH = PROJECT_ROOT / "data" / "paddydataset.csv"
PROCESSED_DIR = PROJECT_ROOT / "outputs" / "processed"
TABLE_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"

## Run Preprocessing

The yield column is excluded from clustering features and saved separately for interpretation.

In [2]:
df = load_paddy_dataset(DATA_PATH)
preprocessing_result = preprocess_paddy_dataset(df)
save_preprocessing_outputs(preprocessing_result, PROCESSED_DIR, TABLE_DIR)

print(preprocessing_result.features.shape)
preprocessing_result.metadata

(2789, 71)


{'rows': 2789,
 'raw_columns': 45,
 'feature_columns_before_encoding': 44,
 'numeric_columns': ['Hectares',
  'Seedrate(in Kg)',
  'LP_Mainfield(in Tonnes)',
  'Nursery area (Cents)',
  'LP_nurseryarea(in Tonnes)',
  'DAP_20days',
  'Weed28D_thiobencarb',
  'Urea_40Days',
  'Potassh_50Days',
  'Micronutrients_70Days',
  'Pest_60Day(in ml)',
  '30DRain( in mm)',
  '30DAI(in mm)',
  '30_50DRain( in mm)',
  '30_50DAI(in mm)',
  '51_70DRain(in mm)',
  '51_70AI(in mm)',
  '71_105DRain(in mm)',
  '71_105DAI(in mm)',
  'Min temp_D1_D30',
  'Max temp_D1_D30',
  'Min temp_D31_D60',
  'Max temp_D31_D60',
  'Min temp_D61_D90',
  'Max temp_D61_D90',
  'Min temp_D91_D120',
  'Max temp_D91_D120',
  'Inst Wind Speed_D1_D30(in Knots)',
  'Inst Wind Speed_D31_D60(in Knots)',
  'Inst Wind Speed_D61_D90(in Knots)',
  'Inst Wind Speed_D91_D120(in Knots)',
  'Relative Humidity_D1_D30',
  'Relative Humidity_D31_D60',
  'Relative Humidity_D61_D90',
  'Relative Humidity_D91_D120',
  'Trash(in bundles)'],
 'ca

## Fit PCA

PCA is fitted on the scaled numeric and one-hot encoded categorical feature matrix.

In [3]:
pca, pca_coordinates, explained_variance, pca_metadata = fit_pca(
    preprocessing_result.features,
    variance_threshold=0.95,
)
save_pca_outputs(
    coordinates=pca_coordinates,
    explained_variance=explained_variance,
    metadata=pca_metadata,
    output_dir=PROCESSED_DIR,
    table_dir=TABLE_DIR,
    figure_dir=FIGURE_DIR,
)

pca_metadata

{'input_rows': 2789,
 'input_features': 71,
 'variance_threshold': 0.95,
 'components_for_threshold': 6,
 'pc1_explained_variance': 0.2895285446001467,
 'pc2_explained_variance': 0.286507005952337,
 'pc1_pc2_cumulative_variance': 0.5760355505524837}

## Explained Variance

In [4]:
explained_variance.head(12)

,component,explained_variance_ratio,cumulative_explained_variance,eigenvalue
0,PC1,2.895285e-01,0.289529,1.207090e+01
1,PC2,2.865070e-01,0.576036,1.194493e+01
2,PC3,1.405013e-01,0.716537,5.857722e+00
3,PC4,9.980820e-02,0.816345,4.161161e+00
4,PC5,9.732281e-02,0.913668,4.057541e+00
5,PC6,4.544208e-02,0.959110,1.894552e+00
6,PC7,1.228840e-02,0.971398,5.123227e-01
7,PC8,1.145276e-02,0.982851,4.774835e-01
8,PC9,1.003119e-02,0.992882,4.182160e-01
9,PC10,6.951496e-03,0.999834,2.898188e-01


## PCA Coordinates Preview

In [5]:
pca_coordinates[["PC1", "PC2", "PC3", "PC4", "PC5", "PC6"]].head()

,PC1,PC2,PC3,PC4,PC5,PC6
0,-2.842845,6.153110,0.159155,2.710404,-2.133793,-1.370596
1,-3.473194,6.462126,2.187628,-2.366672,0.592024,1.062891
2,5.487815,4.571524,0.699258,-0.385303,-3.391927,1.410954
3,2.713493,4.977510,-2.052625,-3.052944,-0.206005,-2.249794
4,4.144343,4.981105,1.833569,1.420143,2.465383,-0.316144


## Saved Outputs

In [8]:
print("Processed outputs:")
for path in sorted(PROCESSED_DIR.glob("*.csv")):
    print(path.relative_to(PROJECT_ROOT))

print("PCA tables:")
for path in sorted(TABLE_DIR.glob("pca*")):
    print(path.relative_to(PROJECT_ROOT))

print("PCA figures:")
for path in sorted(FIGURE_DIR.glob("pca*.png")):
    print(path.relative_to(PROJECT_ROOT))

Processed outputs:
outputs/processed/pca_coordinates_2d.csv
outputs/processed/pca_coordinates_all_components.csv
outputs/processed/processed_features.csv
outputs/processed/similarity_graph_edges.csv
outputs/processed/target_yield.csv
PCA tables:
outputs/tables/pca_explained_variance.csv
outputs/tables/pca_metadata.json
PCA figures:
outputs/figures/pca_2d_projection.png
outputs/figures/pca_2d_projection_colored_by_yield.png
outputs/figures/pca_cumulative_explained_variance.png
outputs/figures/pca_explained_variance_first_20.png
